# State Data Overview
California's state government collects revenue into the **General Fund** (flexible, mostly income and sales tax) and **special funds** (earmarked, e.g. gas taxes for roads). Spending also includes bond funds and federal funds.

**Sources** (all fetched by `scripts/fetch_*.py`, listed with hashes in `data/raw/MANIFEST.csv`):
- Legislative Analyst's Office (LAO) historical revenues, 1950-51 to present, and historical expenditures, 1984-85 to present. These come from Department of Finance and State Controller data.
- BLS CPI-U (via FRED) for inflation, and FRED/Census California population.

**Caveats**
- The revenue series has a break between 2015-16 and 2016-17: LAO publishes the two periods under different accounting categories, so levels shift slightly. Charts mark the break.
- 2018-19 personal income and sales tax figures in LAO's file are misaligned; those cells are replaced with DOF Schedule 3 totals (all funds). Every comparison and repair is logged in `data/processed/revenue_reconciliation.csv`.
- The newest years (shaded "estimate") are budget estimates and proposals, not closed books.
- Real dollars use CPI-U (US city average) and the latest complete fiscal year as the base.

In [1]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from ca_taxes.io import PROCESSED
from ca_taxes.deflate import add_views, base_fy, LAST_ACTUAL_FY
from ca_taxes.style import stacked_area_toggle, base_layout, PALETTE, SOURCE_NOTE
pio.renderers.default = "plotly_mimetype"
rev = pd.read_csv(PROCESSED / "state_revenue.csv")
exp = pd.read_csv(PROCESSED / "state_expenditures.csv")
mac = pd.read_csv(PROCESSED / "macro.csv")
print(f"Revenue rows: {len(rev):,}, fiscal years {rev.fy.min()-1}-{str(rev.fy.min())[2:]} to {rev.fy.max()-1}-{str(rev.fy.max())[2:]}")
print(f"Expenditure rows: {len(exp):,}, fiscal years {exp.fy.min()-1}-{str(exp.fy.min())[2:]} to {exp.fy.max()-1}-{str(exp.fy.max())[2:]}")
print(f"Real-dollar base: fiscal year {base_fy()}; last non-estimate year: {LAST_ACTUAL_FY}")
tot = rev[rev.group != "Transfers"].groupby("fy").amount_k.sum() / 1e6
print()
print("Total state revenue (general + special funds, excl. transfers), $B nominal")
print(tot.loc[[1951, 1971, 1991, 2011, 2025]].round(1).to_string())

Revenue rows: 6,396, fiscal years 1950-51 to 2026-27
Expenditure rows: 22,189, fiscal years 1984-85 to 2026-27
Real-dollar base: fiscal year 2026; last non-estimate year: 2025

Total state revenue (general + special funds, excl. transfers), $B nominal
fy
1951      1.0
1971      5.9
1991     47.0
2011    122.5
2025    308.5
